# Final submission package, step 1 of 2: neural maps (Kaggle T4)

Run on a Kaggle T4 runtime, then run `reproduce_submission.ipynb` on a **CPU** runtime with this notebook's output attached (or run `pipeline.ipynb`, which runs both). No training.

**Weights come from this repository.** `weights/` holds every file the pipeline loads; the three Mask2Former files are stored as numbered parts under 48 MB. On first use the parts are reassembled and every file is checked against the sha256 in `weights/repo_weights.json` and `weights/reproduction_weights.json`; any mismatch stops the run. No dataset needs to be attached for the weights. (Fallback only: without a `weights/` folder, the loader uses a mounted copy of the weights.)

**Test images.** `filseg.reproduction.paths.find_test_images()` tries the known Kaggle mount layouts of the competition data in order (`/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images` first), then accepts a single other `test_images` directory under `/kaggle/input`; otherwise it stops and lists what it found. The run then checks the 180 file names and every image sha256 against the pinned list.

**Deterministic mode is on by default.** Seeds are fixed, cuDNN autotuning is off, `torch.use_deterministic_algorithms(True)` raises on any nondeterministic CUDA operation (it never falls back silently), `CUBLAS_WORKSPACE_CONFIG=:4096:8` is set before CUDA starts, and images are processed sequentially in sorted order (no DataLoader). Map archives are written with fixed ZIP metadata, so equal maps give byte-equal files. The settings and the runtime (PyTorch, CUDA, cuDNN, driver, GPU) are saved in `inference_result.json`. Pass `--nondeterministic` to switch the mode off.

MF uses its original BF16 dtype; T4 has no native BF16 and the installed PyTorch emulates it. An unsupported operation is an error, never an automatic precision change.

**Expected result.** The five archives have the map-set sha256 `d7aa6a41fafc26b07ce89b306b884f6667ebb68b86289ea4745ee44c0b0d195f` (sha256 of the JSON of the five archive sha256, sorted keys).

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Clone this repository at its release tag (needs Internet), unless this notebook already runs inside a checkout.
SOURCE_TAG = "v1.1"
if not (REPO / "src/filseg/reproduction").is_dir():
    REPO = Path("/kaggle/working/filament-reproduction-source")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Matapanino/filament-segmentation-2026-solution.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_TAG], check=True)
# The launcher installs separate pinned U-Net and MF environments in fresh processes.
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))  # the editable install is seen only by new processes; the next cell imports filseg here

In [ ]:
from filseg.reproduction.paths import find_test_images
IMAGE_ROOT = find_test_images()  # or set IMAGE_ROOT = Path("<the test JPEG directory>")
OUTPUT = Path("/kaggle/working/maps")
# Weights: the repository's weights/ folder (parts reassembled and hash-checked on first use).
subprocess.run([sys.executable, "-m", "filseg.reproduction", "infer", "--images", str(IMAGE_ROOT), "--output", str(OUTPUT)], check=True)
result = json.loads((OUTPUT / "inference_result.json").read_text())
print({k: result[k] for k in ("status", "images", "elapsed_seconds", "deterministic")})
print({name: row["sha256"] for name, row in result["networks"].items()})